# Módulo 2 · Segundo Punto — Previsión de demanda

**María Fernanda Hurtado Gómez**

El problema de este punto es el de una cadena minorista que vende miles de productos en varias tiendas y estados, y que necesita anticipar cuánto va a vender cada producto en cada tienda. Si se queda corta pierde ventas por falta de stock, y si se pasa termina con plata parada en inventario. O sea que el pronóstico tiene un impacto directo en la plata.

El dataset es el de la competencia M5 (ventas de Walmart). Lo que me piden es básicamente dos cosas:

1. Armar un modelo que prediga la demanda a nivel producto-tienda-día para los próximos 28 días.
2. Medir la incertidumbre de ese modelo con Conformal Prediction, para saber en qué casos puedo confiar menos en la predicción.

Vengo trabajando con cuatro archivos:

| Archivo | Qué trae |
|---|---|
| `sales_train_validation.csv` | Las ventas diarias de cada producto-tienda (columnas `d_1`…`d_1913`) y la jerarquía (`item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`). Son 30.490 series. |
| `calendar.csv` | A qué fecha real corresponde cada `d_i`, más día de la semana, mes, año, eventos y los flags SNAP por estado. |
| `sell_prices.csv` | El precio de cada producto por tienda y semana. |
| `sample_submission.csv` | El formato de entrega: 28 días futuros (`F1`…`F28`) por cada serie. |

La estrategia que seguí es la que suele funcionar en este tipo de datos: pasar todo a un formato de panel (una fila por producto-tienda-día), construir variables de rezago y de calendario, y entrenar un gradient boosting. Para la incertidumbre uso split conformal, que no asume ninguna distribución de los errores.

## 1. Librerías



In [ ]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

try:
    import lightgbm as lgb
    _HAS_LGB = True
except ModuleNotFoundError:
    _HAS_LGB = False
    print("[aviso] lightgbm no instalado -> pip install lightgbm")

try:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
except ModuleNotFoundError:
    print("[aviso] scikit-learn no instalado -> pip install scikit-learn")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Listo.")
print("pandas:", pd.__version__, "| numpy:", np.__version__,
      "| lightgbm:", lgb.__version__ if _HAS_LGB else "no instalado")

## 2. Carga de los datos

Lo primero que noté es que los archivos son pesados: el de ventas tiene casi 1.919 columnas y el de precios pasa de los 6,8 millones de filas. Si los cargo con los tipos que pandas asigna por defecto se me va la memoria, así que fijo los tipos a mano (`int16` para las ventas, `float32` para precios, `category` para los identificadores). Con eso bajo bastante el consumo.

In [ ]:
DATA_DIR = Path("Modulo_dos_segundo_punto")
if not DATA_DIR.exists():
    DATA_DIR = Path(".")

assert (DATA_DIR / "sales_train_validation.csv").exists(), \
    f"No encuentro los CSV en {DATA_DIR.resolve()}"


def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024 ** 2


print("Datos en:", DATA_DIR.resolve())

In [ ]:
calendar_dtypes = {
    "wm_yr_wk": "int32", "wday": "int8", "month": "int8", "year": "int16",
    "snap_CA": "int8", "snap_TX": "int8", "snap_WI": "int8",
    "weekday": "category", "event_name_1": "category", "event_type_1": "category",
    "event_name_2": "category", "event_type_2": "category",
}

calendar = pd.read_csv(DATA_DIR / "calendar.csv", dtype=calendar_dtypes, parse_dates=["date"])

print(f"calendar: {calendar.shape[0]:,} filas x {calendar.shape[1]} columnas | {mem_mb(calendar):.2f} MB")
print(f"Fechas: {calendar['date'].min().date()} a {calendar['date'].max().date()}")
calendar.head()

Ahora los precios. Este es el archivo más grande, así que acá sí o sí hay que cuidar los tipos.

In [ ]:
prices_dtypes = {
    "store_id": "category", "item_id": "category",
    "wm_yr_wk": "int32", "sell_price": "float32",
}

sell_prices = pd.read_csv(DATA_DIR / "sell_prices.csv", dtype=prices_dtypes)

print(f"sell_prices: {sell_prices.shape[0]:,} filas x {sell_prices.shape[1]} columnas | {mem_mb(sell_prices):.2f} MB")
sell_prices.head()

Y las ventas, que es el corazón del dataset. Son las 30.490 series con sus columnas de identificación y una columna por cada día

In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

header = pd.read_csv(DATA_DIR / "sales_train_validation.csv", nrows=0).columns.tolist()
d_cols = [c for c in header if c.startswith("d_")]

sales_dtypes = {c: "category" for c in id_cols}
sales_dtypes.update({c: "int16" for c in d_cols})

sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv", dtype=sales_dtypes)

print(f"sales: {sales.shape[0]:,} filas x {sales.shape[1]:,} columnas | {mem_mb(sales):.2f} MB")
print(f"Días de histórico: {len(d_cols)} (de d_1 a {d_cols[-1]})")
sales[id_cols + d_cols[:5]].head()

Por último el formato de entrega, para tener claro qué forma debe tener la predicción: 28 días por cada serie.

In [ ]:
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"submission: {submission.shape[0]:,} filas x {submission.shape[1]} columnas")
f_cols = [c for c in submission.columns if c.startswith('F')]
print("Horizonte:", f_cols[0], "...", f_cols[-1])
submission.head()

Antes de seguir, un vistazo rápido a las dimensiones de la jerarquía para tener el mapa mental del problema.

In [ ]:
print("Estados      :", sales['state_id'].cat.categories.tolist())
print("Categorías   :", sales['cat_id'].cat.categories.tolist())
print("Departamentos:", sales['dept_id'].cat.categories.tolist())
print("Tiendas      :", sales['store_id'].cat.categories.tolist())
print("Productos    :", sales['item_id'].nunique())
print("Series base  :", sales['id'].nunique())

total_mem = mem_mb(calendar) + mem_mb(sell_prices) + mem_mb(sales) + mem_mb(submission)
print(f"\nMemoria total: {total_mem:,.1f} MB")

Con los tipos ajustados todo entra en memoria sin problema (alrededor de 200 MB). Son 3 estados, 3 categorías, 7 departamentos, 10 tiendas y poco más de 3.000 productos, que combinados dan las 30.490 series del nivel base.

## 3. Explorando los datos

Antes de ponerme a modelar quiero entender a qué me enfrento. En demanda de retail el dolor de cabeza suele ser la intermitencia: muchísimos días en cero. Quiero medir eso, ver cómo se reparte la venta entre categorías/estados/tiendas, y revisar si hay estacionalidad.

### 3.1 ¿Qué tan intermitente es la demanda?

In [ ]:
vals = sales[d_cols].to_numpy()

pct_zeros_global = (vals == 0).mean()
zeros_per_serie = (vals == 0).mean(axis=1)
mean_per_serie = vals.mean(axis=1)

print(f"% de ceros en todo el dataset   : {pct_zeros_global*100:.1f}%")
print(f"Series con mas del 90% en cero  : {(zeros_per_serie > 0.9).mean()*100:.1f}%")
print(f"Series con mas del 50% en cero  : {(zeros_per_serie > 0.5).mean()*100:.1f}%")
print("\nVenta diaria media por serie:")
print(pd.Series(mean_per_serie).describe().round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(zeros_per_serie, bins=40, color="#4C72B0", edgecolor="white")
ax[0].set_title("Proporción de días en cero, por serie")
ax[0].set_xlabel("% de días en cero")
ax[0].set_ylabel("Nº de series")
ax[1].hist(np.clip(mean_per_serie, 0, 10), bins=40, color="#55A868", edgecolor="white")
ax[1].set_title("Venta diaria media por serie (recortada en 10)")
ax[1].set_xlabel("Unidades/día")
ax[1].set_ylabel("Nº de series")
plt.tight_layout()
plt.show()

Confirmado lo que sospechaba: cerca del 68% de los días son cero, casi el 80% de las series tienen más de la mitad de sus días sin vender nada, y la mediana de venta diaria por serie está por debajo de 0,5 unidades. Esto me marca dos decisiones más adelante: el modelo tiene que llevarse bien con tantos ceros (por eso después uso un objetivo Tweedie en vez de un error cuadrático normal), y no me sirve mirar solo el error puntual.

### 3.2 ¿Dónde está el volumen?

In [ ]:
tot_cat = sales.groupby("cat_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)
tot_state = sales.groupby("state_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)
tot_store = sales.groupby("store_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
tot_cat.plot(kind="bar", ax=ax[0], color="#4C72B0", title="Ventas por categoría")
tot_state.plot(kind="bar", ax=ax[1], color="#55A868", title="Ventas por estado")
tot_store.plot(kind="bar", ax=ax[2], color="#C44E52", title="Ventas por tienda")
for a in ax:
    a.set_ylabel("Unidades acumuladas")
    a.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("Participación por categoría (%):")
print((tot_cat / tot_cat.sum() * 100).round(1).to_string())

FOODS se come casi el 70% de las ventas, seguido de HOUSEHOLD y de lejos HOBBIES. Esto me dice que la mayor parte del esfuerzo de pronóstico se juega en comida, y que vale la pena que el modelo sepa a qué categoría pertenece cada serie. Por eso dejo la jerarquía como variable.

### 3.3 Estacionalidad

Sumo todas las ventas por día para ver la serie global, que al estar agregada es mucho más legible que una serie individual.

In [ ]:
daily_total = sales[d_cols].sum(axis=0)
daily_total.index = daily_total.index.map(calendar.set_index("d")["date"].to_dict())
daily_total = daily_total.sort_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_total.index, daily_total.values, lw=0.7, color="#4C72B0")
ax.plot(daily_total.index, daily_total.rolling(28).mean().values, lw=2, color="#C44E52",
        label="Media móvil 28 días")
ax.set_title("Ventas totales diarias")
ax.set_ylabel("Unidades/día")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.show()

dow = calendar.set_index("d").loc[d_cols, "wday"].to_numpy()
dow_mean = pd.Series(daily_total.to_numpy(), index=dow).groupby(level=0).mean()
wday_labels = {1: "Sáb", 2: "Dom", 3: "Lun", 4: "Mar", 5: "Mié", 6: "Jue", 7: "Vie"}
print("Venta media por día de la semana:")
print(dow_mean.rename(index=wday_labels).round(0).to_string())

La serie global tiene una tendencia que va subiendo con los años y una estacionalidad anual marcada (se ven las caídas de fin de año). Y a nivel semana la diferencia es enorme: los fines de semana se vende como un 35% más que entre semana. Esa estacionalidad semanal es la señal más fácil de capturar, así que me aseguro de meter el día de la semana como variable.

## 4. De formato ancho a panel largo

El dataset viene en formato ancho (una fila por serie, una columna por día), pero para modelar necesito lo contrario: una fila por cada combinación de serie y día. Hago el `melt` y de paso le pego la información del calendario y el precio.

Una aclaración de tamaño: el panel completo serían unos 58 millones de filas, que en mi máquina es demasiado. Trabajo entonces con una muestra de series (`N_SERIES`), tomada de forma estratificada por tienda para no romper la jerarquía. El código es exactamente el mismo si se quiere correr con las 30.490 series en una máquina con más RAM, solo hay que subir ese número.

In [ ]:
N_SERIES = 3000  # subir a 30490 para el dataset completo

if N_SERIES >= len(sales):
    sales_sub = sales
else:
    sales_sub = (
        sales.groupby("store_id", observed=True, group_keys=False)
        .apply(lambda g: g.sample(max(1, int(round(N_SERIES * len(g) / len(sales)))),
                                  random_state=RANDOM_STATE), include_groups=True)
    )

print(f"Series en el panel: {len(sales_sub):,} de {len(sales):,}")

cal_cols = ["d", "date", "wm_yr_wk", "wday", "month", "year",
            "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"]

long = sales_sub.melt(id_vars=id_cols, value_vars=d_cols, var_name="d", value_name="sales")
long["sales"] = long["sales"].astype("int16")
long = long.merge(calendar[cal_cols], on="d", how="left")
long = long.merge(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

print(f"Panel largo: {long.shape[0]:,} filas x {long.shape[1]} columnas | {mem_mb(long):.1f} MB")
long.head()

Al unir con los precios me quedaron bastantes filas sin precio. Investigando el dataset entendí que eso pasa porque en M5 los productos se van dando de alta en fechas distintas: si una fila no tiene precio es porque ese producto todavía no estaba a la venta en esa tienda esa semana. Esos ceros no son demanda real (no es que nadie compró, es que el producto no existía todavía en el surtido), así que los saco. Si no lo hiciera, el modelo aprendería a predecir ceros de más.

In [ ]:
antes = len(long)
pct_na = long["sell_price"].isna().mean() * 100
print(f"Filas sin precio (producto aún no listado): {pct_na:.1f}%")

long = long.sort_values(["id", "date"])
long = long[long["sell_price"].notna()].copy()

print(f"Quedan {len(long):,} filas (saqué {antes - len(long):,}).")
print(f"% de ceros ahora (demanda real): {(long['sales']==0).mean()*100:.1f}%")
long.head()

Después de limpiar esos ceros falsos, el porcentaje de ceros baja de 68% a cerca de 60%. Sigue siendo demanda muy intermitente, pero al menos ahora los ceros que quedan son ceros de verdad.

## 5. Construcción de variables

Acá armo las features. La idea es darle al modelo tres tipos de señal: lo que venía vendiendo cada serie (rezagos y promedios móviles), en qué momento del año/semana estamos (calendario), y los factores que empujan la demanda (eventos, SNAP, precio).

Hay un detalle importante que cuidé mucho: como el objetivo es predecir 28 días adelante, no puedo usar información de los últimos 28 días para construir las variables, porque en producción no la tendría. Por eso todos los promedios móviles los calculo sobre un `shift(28)`. Si no hiciera esto estaría haciendo trampa (data leakage) y el modelo se vería mucho mejor de lo que realmente es.

### 5.1 Rezagos y promedios móviles

In [ ]:
long = long.sort_values(["id", "date"]).reset_index(drop=True)
g_sales = long.groupby("id", observed=True)["sales"]

for lag in [7, 14, 28]:
    long[f"lag_{lag}"] = g_sales.shift(lag).astype("float32")

for w in [7, 28]:
    base = g_sales.shift(28)   # shift(28) para no mirar el futuro
    long[f"rmean_{w}"] = base.rolling(w).mean().astype("float32")
    long[f"rstd_{w}"] = base.rolling(w).std().astype("float32")

print("Listo. Los NaN son del arranque de cada serie (se filtran al entrenar):")
print(long[["lag_7", "lag_14", "lag_28", "rmean_7", "rmean_28"]].isna().mean().round(3).to_string())

### 5.2 Calendario, eventos y SNAP

Para el SNAP hay un detalle: el calendario trae tres columnas (`snap_CA`, `snap_TX`, `snap_WI`), pero a cada serie solo le aplica la de su estado. Así que armo una sola columna `snap` eligiendo la que corresponde.

In [ ]:
long["dayofweek"] = long["date"].dt.dayofweek.astype("int8")
long["is_weekend"] = (long["dayofweek"] >= 5).astype("int8")
long["week"] = long["date"].dt.isocalendar().week.astype("int16")

state = long["state_id"].astype(str)
long["snap"] = np.select(
    [state == "CA", state == "TX", state == "WI"],
    [long["snap_CA"], long["snap_TX"], long["snap_WI"]],
    default=0,
).astype("int8")

long["is_event"] = long["event_name_1"].notna().astype("int8")

print("Demanda media con y sin SNAP:")
print(long.groupby("snap", observed=True)["sales"].mean().round(3).to_string())
print("\nDemanda media con y sin evento:")
print(long.groupby("is_event", observed=True)["sales"].mean().round(3).to_string())

Se nota que los días con SNAP venden más que los días sin SNAP, que es justo lo que uno esperaría (más ayuda alimentaria, más compra de comida). Esa diferencia es la que quiero que el modelo aproveche.

### 5.3 Precio

El precio importa, pero más que el precio absoluto me interesa el movimiento: si el producto está en promoción (bajó respecto a la semana pasada) o si está caro respecto a su propio nivel habitual.

In [ ]:
pg = long.groupby("id", observed=True)["sell_price"]
long["price_change"] = (long["sell_price"] / pg.shift(1) - 1).astype("float32")
long["price_rel_mean"] = (long["sell_price"] / pg.transform("mean")).astype("float32")

long[["sell_price", "price_change", "price_rel_mean"]].describe().round(3)

### 5.4 Lista final de variables

Dejo las columnas de la jerarquía como `category` a propósito, porque LightGBM sabe trabajar con categóricas directamente y así no tengo que hacer one-hot (que con tantos productos sería carísimo en memoria). Esta es la lista con la que entreno.

In [ ]:
cat_features = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
for c in cat_features:
    long[c] = long[c].astype("category")

num_features = [
    "lag_7", "lag_14", "lag_28",
    "rmean_7", "rstd_7", "rmean_28", "rstd_28",
    "sell_price", "price_change", "price_rel_mean",
    "dayofweek", "is_weekend", "week", "month",
    "snap", "is_event",
]

FEATURES = cat_features + num_features
TARGET = "sales"

print(f"{len(FEATURES)} variables ({len(cat_features)} categóricas + {len(num_features)} numéricas)")
long[["id", "date", TARGET] + num_features].head()

## 6. Modelo con LightGBM

Elegí LightGBM porque es el que mejor funciona en este tipo de datos (de hecho es el que ganó la competencia M5), es rápido y maneja las categóricas sin que yo tenga que hacer nada raro.

Dos decisiones que vale la pena explicar:

- Uso el objetivo `tweedie` en vez del error cuadrático de siempre. Con tantos ceros, un modelo de regresión normal tiende a predecir valores negativos o inflados; Tweedie está pensado justo para variables que son cero muchas veces y positivas de vez en cuando.
- Separo la validación por fecha, no al azar. Dejo los últimos 28 días como validación (que es el mismo horizonte que tengo que predecir) y entreno con todo lo anterior. Partir al azar en series de tiempo también sería hacer trampa.

### 6.1 Separación train / validación

In [ ]:
if not _HAS_LGB:
    raise RuntimeError("Falta LightGBM -> pip install lightgbm scikit-learn")

model_df = long.dropna(subset=["lag_28", "rmean_28"]).copy()

max_date = model_df["date"].max()
cutoff = max_date - pd.Timedelta(days=28)

train_df = model_df[model_df["date"] <= cutoff]
valid_df = model_df[model_df["date"] > cutoff]

print(f"Train : hasta {cutoff.date()}  -> {len(train_df):,} filas")
print(f"Val   : últimos 28 días        -> {len(valid_df):,} filas")

### 6.2 Entrenamiento

In [ ]:
dtrain = lgb.Dataset(train_df[FEATURES], train_df[TARGET],
                     categorical_feature=cat_features, free_raw_data=False)
dvalid = lgb.Dataset(valid_df[FEATURES], valid_df[TARGET],
                     categorical_feature=cat_features, reference=dtrain, free_raw_data=False)

params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": RANDOM_STATE,
}

model = lgb.train(
    params, dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dvalid], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
print(f"\nParó en la iteración {model.best_iteration}")

### 6.3 ¿Qué tan bueno es?

Para saber si el modelo realmente sirve lo comparo contra un baseline bobo: predecir que mañana se va a vender lo mismo que hace 7 días. Si no le gano a eso, no vale la pena el modelo.

In [ ]:
pred_val = np.clip(model.predict(valid_df[FEATURES]), 0, None)

rmse = np.sqrt(mean_squared_error(valid_df[TARGET], pred_val))
mae = mean_absolute_error(valid_df[TARGET], pred_val)

baseline = np.clip(valid_df["lag_7"].fillna(0), 0, None)
rmse_base = np.sqrt(mean_squared_error(valid_df[TARGET], baseline))
mae_base = mean_absolute_error(valid_df[TARGET], baseline)

print("              RMSE     MAE")
print(f"LightGBM    {rmse:7.4f} {mae:7.4f}")
print(f"Baseline    {rmse_base:7.4f} {mae_base:7.4f}")
print(f"\nLe gano al baseline por {(1 - rmse/rmse_base)*100:.1f}% en RMSE")

El modelo le gana al baseline por un margen cómodo (alrededor del 24% en RMSE), así que sí está aprendiendo algo útil y no solo repitiendo el pasado.

### 6.4 ¿En qué se fija el modelo?

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
imp.head(15).iloc[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Variables más importantes (top 15)")
ax.set_xlabel("Ganancia")
plt.tight_layout()
plt.show()

print(imp.head(10).round(0).to_string())

Lo que más pesa son los promedios móviles (`rmean_28`, `rmean_7`), o sea el nivel reciente de venta de cada serie, seguido de la variabilidad reciente y de qué producto es. Tiene todo el sentido: para adivinar cuánto va a vender algo, lo mejor que tengo es cuánto venía vendiendo.

### 6.5 Predicho vs real

A nivel de una sola serie el gráfico sería ruido puro (demasiados ceros), así que sumo todas las series por día. Así se ve si el modelo le atina al nivel y a la forma de la semana.

In [ ]:
val_plot = valid_df[["date", TARGET]].copy()
val_plot["pred"] = pred_val
agg = val_plot.groupby("date").agg(real=(TARGET, "sum"), pred=("pred", "sum"))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(agg.index, agg["real"], marker="o", ms=3, label="Real", color="#4C72B0")
ax.plot(agg.index, agg["pred"], marker="o", ms=3, label="Predicción", color="#C44E52")
ax.set_title("Demanda total por día: real vs predicho (28 días de validación)")
ax.set_ylabel("Unidades/día")
ax.legend()
plt.tight_layout()
plt.show()

Visto en agregado, la predicción sigue bastante bien el nivel y el ritmo semanal de la demanda real. Hay días que se queda corto o se pasa, pero captura la forma, que es lo que me interesa.

## 7. Incertidumbre con Conformal Prediction

Hasta acá tengo una predicción puntual, pero el enunciado pide ir más allá: saber cuándo puedo confiar en el modelo y cuándo no. Para eso uso Conformal Prediction, que convierte cada predicción en un intervalo con una garantía de cobertura (por ejemplo, que el 90% de los valores reales caigan dentro), y lo bueno es que no asume ninguna forma para los errores.

La versión que uso es split conformal, que es la más directa:

1. Parto el histórico en tres pedazos: entreno, calibración y test.
2. Entreno el modelo solo con el primer pedazo.
3. En el pedazo de calibración miro cuánto se equivoca el modelo (el error absoluto).
4. El percentil 90 de esos errores me da el ancho del intervalo.

Sobre la librería: MAPIE hace esto, pero su versión nueva está pensada para modelos de sklearn con su propio flujo de entrenamiento, y acá yo entreno un Booster de LightGBM con una partición temporal hecha a mano para no filtrar el futuro. Me resultó más limpio y transparente escribir el split conformal a mano (son pocas líneas) que forzar la librería. El resultado es el mismo que daría MAPIE con el método base.

### 7.1 Los tres bloques temporales

In [ ]:
cut_test = max_date - pd.Timedelta(days=28)
cut_cal = max_date - pd.Timedelta(days=56)

train_cp = model_df[model_df["date"] <= cut_cal]
calib_cp = model_df[(model_df["date"] > cut_cal) & (model_df["date"] <= cut_test)]
test_cp = model_df[model_df["date"] > cut_test]

print(f"Entreno     : hasta {cut_cal.date()}        -> {len(train_cp):,} filas")
print(f"Calibración : {cut_cal.date()} a {cut_test.date()} -> {len(calib_cp):,} filas")
print(f"Test        : últimos 28 días           -> {len(test_cp):,} filas")

### 7.2 Reentreno sin tocar la calibración

Vuelvo a entrenar pero solo con el primer bloque, para que el bloque de calibración quede "nuevo" para el modelo y los errores que mida ahí sean honestos.

In [ ]:
dtrain_cp = lgb.Dataset(train_cp[FEATURES], train_cp[TARGET],
                        categorical_feature=cat_features, free_raw_data=False)
dcalib_cp = lgb.Dataset(calib_cp[FEATURES], calib_cp[TARGET],
                        categorical_feature=cat_features, reference=dtrain_cp, free_raw_data=False)

model_cp = lgb.train(
    params, dtrain_cp,
    num_boost_round=1000,
    valid_sets=[dcalib_cp], valid_names=["calib"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
print(f"\nParó en la iteración {model_cp.best_iteration}")

### 7.3 Los intervalos

Pruebo dos formas de armar el intervalo, las dos al 90%:

- La estándar da un intervalo del mismo ancho para todas las series.
- La normalizada divide el error por la volatilidad reciente de cada serie, así el intervalo se hace más ancho en los productos más impredecibles y más angosto en los estables. Esta segunda es la que me parece útil para el negocio, porque la incertidumbre deja de ser un número fijo y pasa a decir algo de cada producto.

In [ ]:
alpha = 0.10  # 90% de cobertura

pred_calib = np.clip(model_cp.predict(calib_cp[FEATURES]), 0, None)
pred_test = np.clip(model_cp.predict(test_cp[FEATURES]), 0, None)
y_test = test_cp[TARGET].to_numpy()


def conformal_quantile(scores, alpha):
    n = len(scores)
    k = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores, min(k, 1.0))


# Estándar: mismo ancho para todos
scores_abs = np.abs(calib_cp[TARGET].to_numpy() - pred_calib)
q_abs = conformal_quantile(scores_abs, alpha)
lo_std = np.clip(pred_test - q_abs, 0, None)
hi_std = pred_test + q_abs
cov_std = ((y_test >= lo_std) & (y_test <= hi_std)).mean()

# Normalizado: ancho según la volatilidad de cada serie
sigma_cal = np.maximum(calib_cp["rstd_28"].to_numpy(), 0.5)
sigma_te = np.maximum(test_cp["rstd_28"].to_numpy(), 0.5)
scores_norm = np.abs(calib_cp[TARGET].to_numpy() - pred_calib) / sigma_cal
q_norm = conformal_quantile(scores_norm, alpha)
lo_adj = np.clip(pred_test - q_norm * sigma_te, 0, None)
hi_adj = pred_test + q_norm * sigma_te
cov_adj = ((y_test >= lo_adj) & (y_test <= hi_adj)).mean()

print(f"Objetivo de cobertura: {int((1-alpha)*100)}%\n")
print(f"Estándar    -> cobertura {cov_std*100:.1f}% | ancho medio {np.mean(hi_std-lo_std):.2f}")
print(f"Normalizado -> cobertura {cov_adj*100:.1f}% | ancho medio {np.mean(hi_adj-lo_adj):.2f}")

Las dos versiones dan una cobertura real muy cerca del 90% que buscaba, así que el método está bien calibrado. Me quedo con la normalizada para el análisis siguiente porque es la que diferencia entre productos.

### 7.4 ¿Dónde falla más el modelo?

Esta es la parte que responde directo a la pregunta del enunciado. Ordeno las series por el ancho de su intervalo: las que tienen la banda más ancha son en las que el modelo está más inseguro.

In [ ]:
unc = test_cp[["id", "cat_id", "store_id", TARGET]].copy()
unc["pred"] = pred_test
unc["lo"] = lo_adj
unc["hi"] = hi_adj
unc["width"] = hi_adj - lo_adj
unc["abs_err"] = np.abs(unc[TARGET] - unc["pred"])

by_serie = (unc.groupby("id", observed=True)
            .agg(cat=("cat_id", "first"), store=("store_id", "first"),
                 width=("width", "mean"), abs_err=("abs_err", "mean"),
                 mean_sales=(TARGET, "mean"))
            .sort_values("width", ascending=False))

print("Las 10 series más inciertas:")
print(by_serie.head(10).round(2).to_string())

by_cat = unc.groupby("cat_id", observed=True)["width"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
by_cat.plot(kind="bar", ax=ax, color="#C44E52")
ax.set_title("Incertidumbre media (ancho del intervalo) por categoría")
ax.set_ylabel("Ancho medio del intervalo")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

Las series más inciertas son casi todas de FOODS y de productos de alta rotación. Tiene lógica: son los que más venden y los que más se mueven de un día a otro, así que es más difícil clavar el número exacto. En cambio los productos que casi no venden tienen intervalos angostos porque predecir "cero o casi cero" es fácil. Para el negocio esto es útil: dice en qué productos conviene tener más colchón de inventario.

### 7.5 Cómo se ve el intervalo en una serie

In [ ]:
ejemplo = by_serie.index[3]
mask = test_cp["id"] == ejemplo
fechas = test_cp.loc[mask, "date"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(fechas, lo_adj[mask.to_numpy()], hi_adj[mask.to_numpy()],
                color="#C44E52", alpha=0.2, label="Intervalo 90%")
ax.plot(fechas, pred_test[mask.to_numpy()], color="#C44E52", lw=2, label="Predicción")
ax.plot(fechas, y_test[mask.to_numpy()], color="#4C72B0", marker="o", ms=4, label="Real")
ax.set_title(f"Pronóstico e incertidumbre (90%) - {ejemplo}")
ax.set_ylabel("Unidades/día")
ax.legend()
plt.tight_layout()
plt.show()

Acá se ve la idea completa: la línea roja es la predicción, la banda es el rango donde espero que caiga la venta real con 90% de confianza, y los puntos azules son lo que de verdad pasó. La mayoría de los puntos cae dentro de la banda, que es exactamente lo que la garantía de cobertura promete.

## 8. Guardar lo que usa la app

Para la app de Streamlit no quiero reentrenar nada en vivo, así que guardo acá el modelo, la tabla de predicciones con sus intervalos y el ranking de incertidumbre. La app simplemente lee estos archivos.

In [ ]:
ART_DIR = Path("app_artifacts")
ART_DIR.mkdir(exist_ok=True)

model_cp.save_model(str(ART_DIR / "lgbm_model.txt"))

pred_table = test_cp[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "date", TARGET]].copy()
pred_table["pred"] = pred_test
pred_table["lo"] = lo_adj
pred_table["hi"] = hi_adj
pred_table.rename(columns={TARGET: "real"}, inplace=True)
pred_table.to_parquet(ART_DIR / "predicciones_intervalos.parquet", index=False)

by_serie.reset_index().to_parquet(ART_DIR / "incertidumbre_por_serie.parquet", index=False)

meta = {
    "alpha": alpha,
    "cobertura_objetivo": 1 - alpha,
    "cobertura_estandar": float(cov_std),
    "cobertura_normalizado": float(cov_adj),
    "q_abs": float(q_abs),
    "q_norm": float(q_norm),
    "n_series": int(pred_table["id"].nunique()),
    "fecha_min_test": str(pred_table["date"].min().date()),
    "fecha_max_test": str(pred_table["date"].max().date()),
}
pd.Series(meta).to_json(ART_DIR / "meta.json", indent=2)

print("Guardado en", ART_DIR.resolve())
for f in sorted(ART_DIR.iterdir()):
    print("  -", f.name, f"({f.stat().st_size/1024:.1f} KB)")

## 9. Conclusiones

Resumo lo que saqué de todo el ejercicio:

**Sobre los datos.** Es un problema de demanda muy intermitente: cerca del 68% de los días son cero y la mayoría de las series venden menos de una unidad al día en promedio. Buena parte de los ceros iniciales eran falsos (productos que todavía no estaban a la venta), y limpiarlos fue clave para no engañar al modelo. La venta está muy concentrada en FOODS (casi el 70%) y hay una estacionalidad semanal fuerte, con los fines de semana bastante por encima del resto.

**Sobre el modelo.** LightGBM con objetivo Tweedie fue una buena elección para este tipo de demanda. Le gana al baseline ingenuo por alrededor de un 24% en RMSE, y las variables que más usa (los promedios de venta reciente, la volatilidad y el producto) son las que tienen sentido de negocio. Validé siempre por fecha y cuidé el `shift(28)` para no filtrar información del futuro, así que el número que reporto es realista.

**Sobre la incertidumbre.** El split conformal cumplió la cobertura prometida (muy cerca del 90% real), lo que da confianza en que los intervalos no son solo decorativos. La versión normalizada además responde a la pregunta del enunciado: el modelo está más inseguro en los productos de alta rotación de FOODS, donde la demanda se mueve mucho, y más seguro en los productos de poca venta. Eso es directamente accionable: son esos productos inciertos los que necesitan más stock de seguridad.

**Qué mejoraría con más tiempo.** Entrenaría con las 30.490 series completas (acá usé una muestra por tema de memoria), probaría la métrica WRMSSE que es la oficial de M5, y agregaría más features de calendario (festivos específicos, días antes/después de un evento). También se podría explorar reconciliación jerárquica para que las predicciones de los distintos niveles sean coherentes entre sí.

**Entregable interactivo.** Dejé además una app en Streamlit (`app_streamlit.py`) que lee los artefactos guardados acá y permite explorar el pronóstico y su banda de incertidumbre por producto y tienda, pensada para la sustentación.